# 03 — Gold Training Pairs | HybriNews (Review First)

**Mục tiêu:** chuyển **Silver đã được chốt** sang dữ liệu Gold có `task`, `sample_id`, trường source/target nguyên bản và biểu diễn cho mô hình. **Notebook này không huấn luyện mô hình.**

**Ba cấu hình đã chốt:**

| Dataset | Task | Ngôn ngữ |
|---|---|---|
| CNN/DailyMail | `normal` | `en` |
| XSum | `super` | `en` |
| VietNews | `super` | `vi` |

**Giới hạn bắt buộc:** Không sử dụng XL-Sum VI; không tạo reference thứ hai từ một reference; không tự sinh nhãn mới; không dùng tập test để chọn mô hình, tinh chỉnh prompt hoặc hyperparameter; không tự ý truncate nguồn; Bronze và Silver được giữ nguyên.

### Quy trình hai giai đoạn

**A — PRECHECK, chưa ghi Gold:** kiểm tra manifest Notebook 02, kiểm định từng dataset, thống kê độ dài, xuất phiếu các cặp có dấu hiệu bất thường + phiếu phê duyệt task. **DỪNG SAU A.**

**Bạn tự xử lý:** mở CSV dưới `reports/gold/precheck/reviews/`, chấm `KEEP`/`DROP` đối với mọi dòng được gắn cờ; điền `YES`, người duyệt, lý do trong `task_approval.csv`.

**B — Sau khi bạn duyệt:** xác minh toàn bộ phiếu, áp dụng DROP được duyệt, ghi Gold **tạm**, đọc lại kiểm định, sau đó chốt ba bộ Gold chính thức.

Khác với Notebook 02: Gold không phát hiện lại near-duplicate và không gán nội dung nguồn sang một split khác. Những giới hạn near-duplicate từ Silver vẫn phải được ghi nhận trong báo cáo.

## 0. Đường dẫn, thông số và quy tắc bất biến

In [ ]:
from pathlib import Path
import os, sys, csv, json, hashlib, shutil
from datetime import datetime, timezone
from collections import Counter

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name.lower() == 'notebooks' else HERE
if not (ROOT / 'notebooks').is_dir():
    raise RuntimeError(f'Mở VS Code từ thư mục CK_BIG chứa notebooks/. CWD: {HERE}')

SILVER = ROOT / 'data' / 'silver'
GOLD = ROOT / 'data' / 'gold'
GOLD_STAGING = ROOT / 'data' / '_gold_staging'
REPORTS = ROOT / 'reports'
GOLD_REPORT = REPORTS / 'gold'
PRECHECK = GOLD_REPORT / 'precheck'
REVIEWS = PRECHECK / 'reviews'
SILVER_COMMIT = REPORTS / 'silver' / 'after_review' / 'silver_commit_manifest.json'
SPLITS = ('train', 'validation', 'test')
TASKS = {
    'cnn_dailymail': {'task':'normal', 'language':'en'},
    'xsum': {'task':'super', 'language':'en'},
    'vietnews': {'task':'super', 'language':'vi'},
}

# Chỉ gắn cờ kiểm tra, KHÔNG tự loại theo độ dài.
MIN_SOURCE_UNITS = 20
MIN_TARGET_UNITS = 4
MAX_SOURCE_CHARS = 60000
MAX_TARGET_CHARS = 1500
MAX_REVIEW_ROWS_PER_DATASET = 5000
PREVIEW_SOURCE_CHARS = 900
PREVIEW_TARGET_CHARS = 500

for p in (PRECHECK, REVIEWS, GOLD_REPORT):
    p.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def write_json_new(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('x', encoding='utf-8') as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

def read_csv(path):
    with Path(path).open('r', newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

def write_csv_new(path, rows, fields):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('x',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

def safe_snapshot(folder):
    """Inventory size + mtime: phát hiện thay đổi file Silver sau lúc kiểm định A.
    Không dùng làm checksum mật mã của nội dung parquet."""
    folder=Path(folder)
    paths=sorted(folder.rglob('*.parquet'))
    if not paths:
        raise FileNotFoundError(f'Không có Parquet: {folder}')
    return [{'path':str(p.relative_to(ROOT).as_posix()),
             'bytes':p.stat().st_size, 'mtime_ns':p.stat().st_mtime_ns} for p in paths]

print('ROOT:',ROOT)
print('Python:',sys.version.split()[0])
print('Các task:',TASKS)
print('Silver manifest:',SILVER_COMMIT)

## 0.1. Khởi tạo PySpark Windows (chỉ cần chạy lại cell cấu hình sau Restart Kernel)

In [ ]:
# HADOOP_HOME phải được thiết lập trước khi tạo SparkSession.
# Nếu bạn đã cài winutils Hadoop 3.4.2 trong tài khoản Windows,
# khi VS Code chưa nhận biến môi trường, có thể khai báo thủ công ở đây:
# os.environ['HADOOP_HOME'] = str(Path.home() / 'hadoop-3.4.2')
# os.environ['PATH'] = str(Path(os.environ['HADOOP_HOME']) / 'bin') + os.pathsep + os.environ['PATH']

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql import Window

SPARK_CORES = 2                  # Cấu hình an toàn cho Windows; tăng sau khi ổn định
SHUFFLE_PARTITIONS = 8
spark = (SparkSession.builder
    .appName('HybriNews-03-Gold')
    .master(f'local[{SPARK_CORES}]')
    .config('spark.driver.host','127.0.0.1')
    .config('spark.driver.bindAddress','127.0.0.1')
    .config('spark.python.worker.reuse','false')
    .config('spark.sql.shuffle.partitions',str(SHUFFLE_PARTITIONS))
    .config('spark.sql.adaptive.enabled','true')
    .config('spark.sql.files.maxPartitionBytes',str(32*1024*1024))
    .config('spark.driver.memory','4g')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:',spark.version,'| Master:',spark.sparkContext.master)
print('HADOOP_HOME:',os.environ.get('HADOOP_HOME'))
print('Spark smoke count:',spark.range(5).count())

## A1 — Đọc Silver commit manifest và khóa phiên bản dữ liệu (KHÔNG tạo Gold)

In [ ]:
assert SILVER_COMMIT.is_file(), ('Không tìm thấy Silver manifest đã chốt: '
    f'{SILVER_COMMIT}. Cần hoàn thành Notebook 02 B5.')
SILVER_MANIFEST = read_json(SILVER_COMMIT)
assert SILVER_MANIFEST.get('status') == 'COMPLETE', 'Silver chưa có trạng thái COMPLETE'
assert set(SILVER_MANIFEST.get('datasets', {})) == set(TASKS), 'Silver không phải đúng ba dataset đã chốt'

for name in TASKS:
    info = SILVER_MANIFEST['datasets'][name]
    # Dùng số liệu từ manifest Silver đã chốt; không hard-code snapshot trước đó.
    assert int(info['rows']) > 0, f'{name}: số dòng manifest không hợp lệ'
    assert (SILVER/name).is_dir(), f'Thiếu thư mục Silver {name}'
    assert sum(int(v) for v in info['split_counts'].values()) == int(info['rows'])
    print(f'{name}: {int(info["rows"]):,} dòng, splits={info["split_counts"]}')

SILVER_MANIFEST_SHA = file_sha256(SILVER_COMMIT)
A_RESULTS = {}
print('Đã khóa manifest Silver:', SILVER_MANIFEST_SHA[:16]+'...')

## A2 — Hàm kiểm định và tạo phiếu rà soát bất thường

**Kiểm tra bắt buộc:** số dòng và split so với Silver manifest; schema; source/target không rỗng; hash và record ID không rỗng; **mỗi source_hash đúng một bản ghi trong từng dataset**. Không gán nhãn khác với ba task đã thống nhất.

**Các cờ thăm dò:** source quá ngắn (dưới 20 đơn vị cách trắng), target quá ngắn (dưới 4), target dài hơn source, ký tự thay thế Unicode `�`, source trên 60.000 ký tự, target trên 1.500 ký tự. **Cờ không đồng nghĩa với lỗi**. Khi có cờ, notebook xuất CSV để bạn tự quyết định `KEEP` hoặc `DROP`.

Nếu số cờ vượt mức bảo vệ 5.000 dòng/dataset, notebook sẽ dừng và yêu cầu điều chỉnh quy trình rà soát thay vì tự lấy mẫu bỏ qua các dòng.

In [ ]:
REQUIRED_SILVER_COLS = {
    'dataset','language','split','article_id','record_id',
    'source_text','target_text','source_hash','target_hash'
}
REVIEW_FIELDS = [
    'dataset','split','article_id','record_id','source_hash','target_hash',
    'flag_reasons','source_units','target_units','source_preview','target_preview',
    'review_decision','review_notes'
]

def quality_expr(df):
    su=F.size(F.split(F.trim(F.col('source_text')),r'\s+'))
    tu=F.size(F.split(F.trim(F.col('target_text')),r'\s+'))
    marked=(df.withColumn('source_units',su).withColumn('target_units',tu)
      .withColumn('flag_reasons',F.concat_ws(';',
        F.when(F.col('source_units')<MIN_SOURCE_UNITS,F.lit('VERY_SHORT_SOURCE')),
        F.when(F.col('target_units')<MIN_TARGET_UNITS,F.lit('VERY_SHORT_TARGET')),
        F.when(F.col('target_units')>F.col('source_units'),F.lit('TARGET_LONGER_THAN_SOURCE')),
        F.when(F.instr('source_text','\ufffd')>0,F.lit('REPLACEMENT_CHAR_SOURCE')),
        F.when(F.instr('target_text','\ufffd')>0,F.lit('REPLACEMENT_CHAR_TARGET')),
        F.when(F.length('source_text')>MAX_SOURCE_CHARS,F.lit('VERY_LONG_SOURCE')),
        F.when(F.length('target_text')>MAX_TARGET_CHARS,F.lit('VERY_LONG_TARGET'))
       )))
    return marked

def check_silver_dataset(name, df):
    missing=REQUIRED_SILVER_COLS-set(df.columns)
    if missing:
        raise RuntimeError(f'{name}: Silver thiếu các cột {sorted(missing)}')
    info=SILVER_MANIFEST['datasets'][name]
    counts={r['split']:r['count'] for r in df.groupBy('split').count().collect()}
    expected={k:int(v) for k,v in info['split_counts'].items()}
    if counts != expected or sum(counts.values()) != int(info['rows']):
        raise RuntimeError(f'{name}: dữ liệu Silver khác commit manifest: {counts} != {expected}')
    invalid=(F.col('dataset').isNull() | (F.col('dataset')!=F.lit(name)) |
             F.col('language').isNull() | (F.col('language')!=F.lit(TASKS[name]['language'])) |
             F.col('split').isNull() | ~F.col('split').isin(*SPLITS) |
             F.col('record_id').isNull() | F.col('source_hash').isNull() |
             F.col('target_hash').isNull() |
             (F.length(F.trim(F.coalesce('source_text',F.lit(''))))==0) |
             (F.length(F.trim(F.coalesce('target_text',F.lit(''))))==0))
    if df.filter(invalid).limit(1).count():
        raise RuntimeError(f'{name}: có bản ghi Silver không hợp lệ, dừng trước Gold')
    dup=(df.groupBy('source_hash').count().filter(F.col('count')>1).limit(1).count())
    if dup:
        raise RuntimeError(f'{name}: có source_hash trùng, kiểm tra lại Silver')
    return counts

def run_precheck(name):
    if name in A_RESULTS:
        raise RuntimeError(f'{name}: đã precheck trong kernel này; không chạy lặp')
    inventory=safe_snapshot(SILVER/name)
    df=spark.read.parquet(str(SILVER/name))
    counts=check_silver_dataset(name, df)
    marked=quality_expr(df)
    flag_df=marked.filter(F.length('flag_reasons')>0)
    n_flags=flag_df.count()
    if n_flags>MAX_REVIEW_ROWS_PER_DATASET:
        raise RuntimeError(f'{name}: {n_flags:,} cờ > {MAX_REVIEW_ROWS_PER_DATASET:,}. '
            'Chưa xuất đủ phiếu: điều chỉnh chiến lược rà soát rồi chạy lại, không tự bỏ qua.')
    flagged=(flag_df.select(
        'dataset','split','article_id','record_id','source_hash','target_hash',
        'flag_reasons','source_units','target_units',
        F.substring('source_text',1,PREVIEW_SOURCE_CHARS).alias('source_preview'),
        F.substring('target_text',1,PREVIEW_TARGET_CHARS).alias('target_preview'))
        .orderBy('record_id').collect())
    review_rows=[{**r.asDict(), 'review_decision':'', 'review_notes':''} for r in flagged]
    review_path=REVIEWS/f'{name}_risk_review.csv'
    if review_path.exists():
        previous=read_csv(review_path)
        now={r['record_id'] for r in review_rows}
        if {r['record_id'] for r in previous}!=now or len(previous)!=len(review_rows):
            raise RuntimeError(f'{review_path} đã khác nguồn Silver; không ghi đè phiếu cũ')
        print(f'[{name}] Giữ nguyên phiếu review đã tồn tại')
    else:
        write_csv_new(review_path,review_rows,REVIEW_FIELDS)
    stats=(marked.groupBy('split').agg(
        F.count('*').alias('rows'),
        F.expr('percentile_approx(source_units, array(0.5,0.95), 10000)').alias('src_q'),
        F.expr('percentile_approx(target_units, array(0.5,0.95), 10000)').alias('tgt_q'),
        F.avg('source_units').alias('mean_source_units'),
        F.avg('target_units').alias('mean_target_units')).collect())
    length_rows=[]
    for r in stats:
        length_rows.append({
            'dataset':name,'split':r['split'],'rows':r['rows'],
            'source_p50_units':r['src_q'][0],'source_p95_units':r['src_q'][1],
            'target_p50_units':r['tgt_q'][0],'target_p95_units':r['tgt_q'][1],
            'mean_source_units':round(r['mean_source_units'],2),
            'mean_target_units':round(r['mean_target_units'],2)})
    # Ảnh xem trước nhẹ; không dùng mẫu test cho lựa chọn mô hình.
    previews=(df.filter(F.col('split')!='test')
        .select('dataset','split','record_id',
            F.substring('source_text',1,350).alias('source_preview'),
            F.substring('target_text',1,250).alias('target_preview'))
        .orderBy('record_id').limit(12).collect())
    A_RESULTS[name]={
        'dataset':name,'silver_rows':sum(counts.values()),'by_split':counts,
        'flags':n_flags,'flag_record_ids':sorted(r['record_id'] for r in review_rows),
        'parquet_inventory':inventory,'review_csv':str(review_path.relative_to(ROOT)),
        'length_rows':length_rows,
        'previews':[r.asDict() for r in previews]
    }
    print(f'[{name}] ĐẠT kiểm tra Silver: {sum(counts.values()):,} bản ghi; '
          f'{n_flags:,} bản ghi cần xem xét; chưa ghi Gold.')
    print('Split:',counts)
    return A_RESULTS[name]

### CNN/DailyMail — A: kiểm tra và xuất phiếu

In [ ]:
run_precheck('cnn_dailymail')

### XSum — A: kiểm tra và xuất phiếu

In [ ]:
run_precheck('xsum')

### VietNews — A: kiểm tra và xuất phiếu (giữ dấu `_`)

In [ ]:
run_precheck('vietnews')

## A3 — Tổng hợp kết quả, xuất task approval và **DỪNG ĐỂ BẠN XEM**

**Hãy mở và xem trước** `reports/gold/precheck/quality_summary.csv`, `length_statistics.csv`, `train_val_preview.csv` và ba file `reviews/*_risk_review.csv`.

- Với từng dòng trong file `*_risk_review.csv`, nhập `review_decision=KEEP` hoặc `DROP` sau khi đọc source/target (các cột preview có thể bị cắt ngắn; nếu chưa đủ, đối chiếu bản ghi Silver bằng `record_id`). Nếu file chỉ có header thì không cần chấm.
- Với `task_approval.csv`, giữ đúng mapping đã thống nhất; điền `approved=YES`, `approved_by`, `approval_reason` sau khi xác nhận.
- Dữ liệu Reference Suitability trước đó có nhãn **AI-assisted preliminary**, nên phiếu phê duyệt này chỉ xác nhận *lựa chọn phạm vi/nhiệm vụ*; không tự biến kết quả sơ bộ thành “Human Audit hoàn tất”.
- **Dừng sau A3.** Không chạy các cell B trước khi chấm và duyệt xong.

In [ ]:
if set(A_RESULTS)!=set(TASKS):
    raise RuntimeError('Chưa kiểm định xong đủ CNN/DailyMail, XSum, VietNews.')

quality=[{'dataset':name,'task':TASKS[name]['task'],
          'silver_rows':A_RESULTS[name]['silver_rows'],
          'risk_review_rows':A_RESULTS[name]['flags'],
          **{f'{split}_rows':A_RESULTS[name]['by_split'].get(split,0) for split in SPLITS}}
         for name in TASKS]
lengths=[row for name in TASKS for row in A_RESULTS[name]['length_rows']]
previews=[row for name in TASKS for row in A_RESULTS[name]['previews']]

files={
    PRECHECK/'quality_summary.csv':(quality,list(quality[0])),
    PRECHECK/'length_statistics.csv':(lengths,list(lengths[0])),
    PRECHECK/'train_val_preview.csv':(previews,list(previews[0])),
}
for path,(rows,cols) in files.items():
    if not path.exists():write_csv_new(path,rows,cols)
    elif read_csv(path)!=[{k:str(v) for k,v in r.items()} for r in rows]:
        # CSV floating-point / number string formats có thể khác nhau giữa các lần chạy.
        print('LƯU Ý: báo cáo đã tồn tại, giữ nguyên:',path.name)

approval_path=PRECHECK/'task_approval.csv'
approval_rows=[{
    'dataset':name,'language':TASKS[name]['language'],'task':TASKS[name]['task'],
    'evidence_status':'EDA_PLUS_PRELIMINARY_AI_ASSISTED_AUDIT',
    'approved':'','approved_by':'','approval_reason':''} for name in TASKS]
if not approval_path.exists():
    write_csv_new(approval_path,approval_rows,list(approval_rows[0]))
else:
    print('Không ghi đè phiếu task_approval.csv đã có')

payload={
    'status':'REVIEW_PENDING',
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'silver_manifest_sha256':SILVER_MANIFEST_SHA,
    'gold_policy':{'task_map':TASKS,'source_min_units':MIN_SOURCE_UNITS,
      'target_min_units':MIN_TARGET_UNITS,'max_source_chars':MAX_SOURCE_CHARS,
      'max_target_chars':MAX_TARGET_CHARS,
      'format':'one_source_one_target','vietnews_segmented_text_preserved':True},
    'datasets':{name:{k:v for k,v in A_RESULTS[name].items() if k not in ('length_rows','previews')} for name in TASKS},
}
precheck_manifest=PRECHECK/'precheck_manifest.json'
if precheck_manifest.exists():
    old=read_json(precheck_manifest)
    old_cmp={k:v for k,v in old.items() if k!='created_at_utc'}
    new_cmp={k:v for k,v in payload.items() if k!='created_at_utc'}
    if old_cmp!=new_cmp:
        raise RuntimeError('Precheck khác snapshot trước! Không ghi đè; kiểm tra Silver đã thay đổi.')
else:
    write_json_new(precheck_manifest,payload)

print('=== KẾT QUẢ A ===')
for row in quality:print(row)
print('Đọc phiếu tại:', REVIEWS)
print('Phê duyệt tại:', approval_path)
print('DỪNG TẠI ĐÂY. Chấm xong rồi mới sang B1.')

### A4 — Tra cứu toàn văn khi preview chưa đủ để quyết định KEEP/DROP

In [ ]:
def show_silver_record(dataset, record_id):
    """Chỉ chạy khi cần đọc đầy đủ một bài được gắn cờ trong phiếu review."""
    if dataset not in TASKS:
        raise ValueError(f'Dataset không hợp lệ: {dataset}')
    rows=(spark.read.parquet(str(SILVER/dataset))
          .filter(F.col('record_id')==F.lit(record_id))
          .select('dataset','split','article_id','record_id','source_text','target_text')
          .limit(2).collect())
    if len(rows)!=1:
        raise RuntimeError(f'Tìm được {len(rows)} bản ghi, cần đúng một record_id')
    row=rows[0]
    print('DATASET:',row['dataset'],'| SPLIT:',row['split'],'| ARTICLE:',row['article_id'])
    print('\n=== FULL SOURCE ===\n',row['source_text'])
    print('\n=== FULL TARGET ===\n',row['target_text'])

# Ví dụ: chỉ bỏ comment khi bạn có record_id trong phiếu review.
# show_silver_record('xsum','PASTE_RECORD_ID_HERE')

---
# GIAI ĐOẠN B — Chỉ chạy SAU KHI bạn duyệt toàn bộ phiếu A

**Nếu Restart Kernel:** chỉ chạy lại các cell `0`, `0.1`, `A1` và các **cell khai báo hàm** (A2, B1, B2); **không gọi lại `run_precheck()`** ở ba dataset, không cần xuất lại CSV.

B1 kiểm tra cứng phiếu task + từng `record_id`; B2 dựng Gold có kiểm định; B3 kiểm tra chung; B4 chốt Gold. **Không có `approved=YES` hoặc còn dòng review trống → dừng.**

## B1 — Đọc phiếu đã chấm; xác nhận task và mọi quyết định KEEP/DROP

In [ ]:
GOLD_PRECHECK_MANIFEST=PRECHECK/'precheck_manifest.json'
GOLD_APPROVAL=PRECHECK/'task_approval.csv'
B_APPROVED={}
B_REVIEW_DIGESTS={}
B_EXPECTED={}
B_BUILT={}

def require_approved():
    if not GOLD_PRECHECK_MANIFEST.is_file():
        raise FileNotFoundError('Chưa có precheck_manifest.json: cần chạy Giai đoạn A trước')
    cfg=read_json(GOLD_PRECHECK_MANIFEST)
    if cfg.get('status')!='REVIEW_PENDING':
        raise RuntimeError('Precheck manifest sai trạng thái.')
    if file_sha256(SILVER_COMMIT)!=cfg['silver_manifest_sha256']:
        raise RuntimeError('Silver manifest đã bị thay đổi sau A. DỪNG')
    if cfg['gold_policy']['task_map'] != TASKS:
        raise RuntimeError('Mapping task thay đổi sau A, cần duyệt lại.')
    if not GOLD_APPROVAL.is_file():
        raise FileNotFoundError('Chưa có task_approval.csv')
    approval=read_csv(GOLD_APPROVAL)
    if len(approval)!=len(TASKS) or {r['dataset'] for r in approval}!=set(TASKS):
        raise RuntimeError('Phiếu task_approval phải có chính xác 3 dataset')
    for row in approval:
        name=row['dataset']
        if (row['task'],row['language'])!=(TASKS[name]['task'],TASKS[name]['language']):
            raise RuntimeError(f'{name}: task/language không đúng mapping đã chốt')
        if row['approved'].strip().upper()!='YES':
            raise RuntimeError(f'{name}: cần approved=YES sau khi bạn đã xem kết quả A')
        if not row['approved_by'].strip() or len(row['approval_reason'].strip())<10:
            raise RuntimeError(f'{name}: cần ghi approved_by và lý do >=10 ký tự')
    approved={}
    digests={'task_approval.csv':file_sha256(GOLD_APPROVAL)}
    for name in TASKS:
        record_expected=cfg['datasets'][name]['flag_record_ids']
        path=REVIEWS/f'{name}_risk_review.csv'
        if not path.is_file():raise FileNotFoundError(path)
        review=read_csv(path)
        ids=[row['record_id'] for row in review]
        if len(ids)!=len(set(ids)) or sorted(ids)!=record_expected:
            raise RuntimeError(f'{name}: phiếu review không khớp danh sách cần chấm từ A')
        decisions={}
        for row in review:
            action=row['review_decision'].strip().upper()
            if action not in ('KEEP','DROP'):
                raise RuntimeError(f'{name}: record_id {row["record_id"][:12]} chưa có KEEP/DROP')
            decisions[row['record_id']]=action
        approved[name]={
            'drop_ids':{rid for rid,dec in decisions.items() if dec=='DROP'},
            'keep_ids':{rid for rid,dec in decisions.items() if dec=='KEEP'},
            'reviewed_count':len(decisions),
            'review_csv_path':str(path),
            'parquet_inventory':cfg['datasets'][name]['parquet_inventory'],
            'silver_rows':int(cfg['datasets'][name]['silver_rows']),
            'by_split':cfg['datasets'][name]['by_split']}
        digests[name+'_risk_review.csv']=file_sha256(path)
    B_APPROVED.clear();B_APPROVED.update(approved)
    B_REVIEW_DIGESTS.clear(); B_REVIEW_DIGESTS.update(digests)
    B_EXPECTED.clear(); B_EXPECTED.update(cfg)
    print('✅ B1 đã duyệt tất cả quyết định. Chưa tạo Gold.')
    for name in TASKS:
        print(f'{name}: {approved[name]["reviewed_count"]} dòng đã chấm; '
              f'DROP={len(approved[name]["drop_ids"])}; KEEP={len(approved[name]["keep_ids"])}')

require_approved()

## B2 — Hàm tạo Gold TẠM theo dataset

**Tất cả nhãn đã chốt từ dataset**; ví dụ XSum có `task=super`. **Không tạo bản tóm tắt mới.**

Gold lưu `source_text`, `target_text` **đúng nguyên dạng Silver**. Với VietNews, giữ `_` trong trường gốc; thêm `model_source_text` và `model_target_text` chuyển `_` thành khoảng trắng để mô hình có tùy chọn đọc tiếng Việt tự nhiên. Các cột `model_*` là **biểu diễn phụ**, không thay thế dữ liệu gốc. Không tokenizer/truncation ở bước Gold.

In [ ]:
def b_check_inputs_unchanged(name):
    if file_sha256(SILVER_COMMIT)!=B_EXPECTED['silver_manifest_sha256']:
        raise RuntimeError('Manifest Silver thay đổi sau phê duyệt')
    p=PRECHECK/'task_approval.csv'
    if file_sha256(p)!=B_REVIEW_DIGESTS['task_approval.csv']:
        raise RuntimeError('Task approval CSV đã bị chỉnh sửa sau B1')
    review=REVIEWS/f'{name}_risk_review.csv'
    if file_sha256(review)!=B_REVIEW_DIGESTS[name+'_risk_review.csv']:
        raise RuntimeError(f'{name}: review CSV đã thay đổi sau B1')
    if safe_snapshot(SILVER/name)!=B_APPROVED[name]['parquet_inventory']:
        raise RuntimeError(f'{name}: Silver Parquet đã thay đổi sau precheck A. DỪNG')

def b_write_gold_staging(name):
    if set(B_APPROVED)!=set(TASKS):
        raise RuntimeError('Phải chạy B1 và hoàn tất approval trước')
    if name in B_BUILT:
        raise RuntimeError(f'{name} đã tạo Gold tạm trong phiên này')
    b_check_inputs_unchanged(name)
    staging=GOLD_STAGING/name
    dest=GOLD/name
    if staging.exists() or dest.exists():
        raise FileExistsError(f'{name}: đã tồn tại Gold tạm/chính thức; KHÔNG ghi đè: {staging} / {dest}')
    source=spark.read.parquet(str(SILVER/name))
    split_before=check_silver_dataset(name,source)
    drops=B_APPROVED[name]['drop_ids']
    if drops:
        schema=T.StructType([T.StructField('record_id',T.StringType(),False)])
        drop_df=spark.createDataFrame([(v,) for v in sorted(drops)],schema)
        clean=source.join(F.broadcast(drop_df),on='record_id',how='left_anti')
    else:
        clean=source
    task=TASKS[name]['task']
    lang=TASKS[name]['language']
    if name=='vietnews':
        model_source=F.regexp_replace(F.col('source_text'),'_',' ')
        model_target=F.regexp_replace(F.col('target_text'),'_',' ')
    else:
        model_source=F.col('source_text')
        model_target=F.col('target_text')
    out=(clean.select(*sorted(REQUIRED_SILVER_COLS))
       .withColumn('task',F.lit(task))
       .withColumn('sample_id',F.sha2(F.concat_ws('|',F.col('dataset'),F.lit(task),F.col('record_id')),256))
       .withColumn('model_source_text',model_source)
       .withColumn('model_target_text',model_target)
       .withColumn('gold_schema_version',F.lit('1.0')))
    staging.parent.mkdir(parents=True,exist_ok=True)
    (out.write.mode('errorifexists').partitionBy('split').parquet(str(staging)))
    verify=spark.read.parquet(str(staging))
    after={r['split']:r['count'] for r in verify.groupBy('split').count().collect()}
    n=sum(after.values())
    expected_n=int(B_APPROVED[name]['silver_rows'])-len(drops)
    if n!=expected_n:
        raise AssertionError(f'{name}: ghi Gold tạm {n} != dự kiến {expected_n}')
    if verify.filter((F.col('task')!=F.lit(task)) |
             (F.length(F.trim(F.coalesce('model_source_text',F.lit(''))))==0) |
             (F.length(F.trim(F.coalesce('model_target_text',F.lit(''))))==0)).limit(1).count():
        raise AssertionError(f'{name}: task hoặc text Gold tạm không hợp lệ')
    if verify.groupBy('source_hash').count().filter(F.col('count')>1).limit(1).count():
        raise AssertionError(f'{name}: Gold còn trùng source_hash')
    B_BUILT[name]={
        'dataset':name,'language':lang,'task':task,
        'silver_rows':B_APPROVED[name]['silver_rows'],
        'review_rows':B_APPROVED[name]['reviewed_count'],
        'review_dropped':len(drops),'gold_rows':n,
        'by_split':after,'staging_directory':str(staging),
        'source_silver_manifest_sha256':B_EXPECTED['silver_manifest_sha256']}
    print(f'✅ {name}: {B_APPROVED[name]["silver_rows"]:,} Silver → {n:,} Gold tạm, '
          f'đã loại {len(drops)} dòng theo phiếu. Split:',after)
    return B_BUILT[name]

### CNN/DailyMail — B: tạo Gold tạm (Normal)

In [ ]:
b_write_gold_staging('cnn_dailymail')

### XSum — B: tạo Gold tạm (Super)

In [ ]:
b_write_gold_staging('xsum')

### VietNews — B: tạo Gold tạm (Super, giữ văn bản đã tách từ)

In [ ]:
b_write_gold_staging('vietnews')

## B3 — Đối chiếu đủ ba Gold TẠM trước khi chốt

In [ ]:
if set(B_BUILT)!=set(TASKS):
    raise RuntimeError(f'Chưa tạo đủ 3 bộ Gold tạm: còn thiếu {set(TASKS)-set(B_BUILT)}')
for name in TASKS:
    b_check_inputs_unchanged(name)
    stats=B_BUILT[name]
    if stats['silver_rows']-stats['review_dropped']!=stats['gold_rows']:
        raise RuntimeError(f'{name}: số dòng Silver/Gold không cân bằng')
    if sum(stats['by_split'].values())!=stats['gold_rows'] or set(stats['by_split'])!=set(SPLITS):
        raise RuntimeError(f'{name}: thiếu split sau khi tạo Gold tạm')
print('=== B3 KIỂM ĐỊNH ĐẠT — CHƯA COMMIT GOLD ===')
for stats in B_BUILT.values():
    print(stats['dataset'], 'Silver=',stats['silver_rows'],
          'DROP=',stats['review_dropped'],'Gold=',stats['gold_rows'],
          'Task=',stats['task'])

## B4 — CHỐT GOLD CHÍNH THỨC (chỉ chạy khi B3 đạt)

Cell này chuyển **ba thư mục Gold tạm đã kiểm tra** sang `data/gold/` rồi ghi `reports/gold/gold_commit_manifest.json` với `status=COMPLETE`. Không xóa và không ghi đè Gold có sẵn. Nếu mất điện hoặc có lỗi giữa chừng, **không coi là hoàn thành nếu chưa có manifest COMPLETE**.

In [ ]:
def b_commit_gold():
    if set(B_BUILT)!=set(TASKS):
        raise RuntimeError('Chưa kiểm tra đủ ba Gold tạm ở B3')
    dest_manifest=GOLD_REPORT/'gold_commit_manifest.json'
    summary_path=GOLD_REPORT/'gold_summary.csv'
    if dest_manifest.exists() or summary_path.exists():
        raise FileExistsError('Đã có manifest hoặc gold_summary.csv: không ghi đè. Kiểm tra bản chạy trước.')
    for name in TASKS:
        b_check_inputs_unchanged(name)
        src=GOLD_STAGING/name
        dst=GOLD/name
        if not src.is_dir() or dst.exists():
            raise RuntimeError(f'{name}: Gold tạm không tồn tại hoặc Gold chính thức đã có: {src}, {dst}')
        actual=spark.read.parquet(str(src)).count()
        if actual!=B_BUILT[name]['gold_rows']:
            raise RuntimeError(f'{name}: Gold tạm đã thay đổi trước commit')
    GOLD.mkdir(parents=True,exist_ok=True)
    committed=[]
    try:
        for name in TASKS:
            src=GOLD_STAGING/name
            dst=GOLD/name
            src.rename(dst)  # Không overwrite, phải ở cùng ổ đĩa
            committed.append(name)
            print(f'✅ ĐÃ CHỐT: {name} → {dst}')
        summary=[]
        for name in TASKS:
            s=B_BUILT[name]
            summary.append({
                'dataset':name,'language':s['language'],'task':s['task'],
                'silver_rows':s['silver_rows'], 'review_dropped':s['review_dropped'],
                'gold_rows':s['gold_rows'],
                **{split:s['by_split'].get(split,0) for split in SPLITS}})
        write_csv_new(GOLD_REPORT/'gold_summary.csv',summary,list(summary[0]))
        manifest={
            'status':'COMPLETE',
            'created_at_utc':datetime.now(timezone.utc).isoformat(),
            'silver_commit_manifest_sha256':B_EXPECTED['silver_manifest_sha256'],
            'precheck_manifest_sha256':file_sha256(GOLD_PRECHECK_MANIFEST),
            'review_files_sha256':B_REVIEW_DIGESTS,
            'datasets':{name:{'task':B_BUILT[name]['task'],
                'language':B_BUILT[name]['language'],'rows':B_BUILT[name]['gold_rows'],
                'split_counts':B_BUILT[name]['by_split'],
                'directory':str(GOLD/name)} for name in TASKS},
            'notes':[
                'One source, one approved target; no new reference generated',
                'No tokenizer or truncation performed in Gold',
                'VietNews source_text retains underscores; model_* replaces underscores with spaces',
                'Human review of source-target suitability must not be overstated if AI-assisted',
                'Near-duplicate audit inherited limitations from Silver blocking'
            ]}
        write_json_new(dest_manifest,manifest)
        print('✅ GOLD ĐÃ HOÀN THÀNH:',dest_manifest)
    except Exception:
        print('❌ Gold commit chưa hoàn tất. Đã chuyển:',committed)
        print('KHÔNG sử dụng Gold nếu manifest chưa có status COMPLETE. Không xóa tự động.')
        raise

b_commit_gold()

---
## Kết thúc Notebook 03 và chuyển sang Training

**Đầu ra Gold chính thức (sau B4):**

```text
CK_BIG/
├── data/
│   ├── silver/                         # GIỮ NGUYÊN
│   └── gold/
│       ├── cnn_dailymail/              # normal
│       ├── xsum/                       # super
│       └── vietnews/                   # super
└── reports/
    └── gold/
        ├── precheck/
        │   ├── quality_summary.csv
        │   ├── length_statistics.csv
        │   ├── train_val_preview.csv
        │   ├── task_approval.csv
        │   ├── reviews/
        │   │   ├── cnn_dailymail_risk_review.csv
        │   │   ├── xsum_risk_review.csv
        │   │   └── vietnews_risk_review.csv
        │   └── precheck_manifest.json
        ├── gold_summary.csv
        └── gold_commit_manifest.json
```

**Lưu ý khoa học:** Tầng Gold đảm bảo schema, split, nhãn nhiệm vụ và kiểm soát các dòng được reviewer quyết định. Nó chưa chứng minh factuality hoàn hảo, chưa đo token của Scratch/Qwen, chưa loại toàn bộ near-duplicate và chưa đánh giá mô hình. Việc chọn max input/output tokens, tokenizer, prompt template, validation và test protocol phải thực hiện riêng trong các notebook huấn luyện. **Tập test chỉ dùng sau khi chốt mô hình/quy trình.**

Sau khi hoàn tất, gửi `reports/gold/gold_summary.csv`, `gold_commit_manifest.json`, ba báo cáo quality/length cùng notebook có output để đối chiếu.